# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ramithnayak8/ML_pipeline/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

A page deserves review first if it shows two or more of these observable weaknesses at once:

- **stale_visible** -- not updated in 180+ days, but still drawing real demand
  (`impressions_90d >= 500`).
- **low_ctr_visible** -- visible and well-positioned (position 1-20) but earning clicks below
  what the signal audit (`w04_signal_audit.ipynb`) showed similar-position pages earn
  (`ctr < 0.5`). That test was CONFIRMED on n=9,759 with a +12.6-point gap, so it earns a spot
  here.
- **thin_visible** -- under 1,200 words while still drawing meaningful demand
  (`impressions_90d >= 250`). The signal audit confirmed word count tracks traffic even within
  one content type, so a thin-but-visible page plausibly has room to grow.

None of the three touches `trend_direction` or `trend_pct` -- the score never sees the label,
only pre-decision signals. Reason codes describe symptoms, not the answer.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_down"] = (df["trend_direction"] == "down").astype(int)  # evaluation only -- never in the score

stale_visible = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
low_ctr_visible = ((df["impressions_90d"] >= 500) & (df["avg_position"] > 0)
                   & (df["avg_position"] <= 20) & (df["ctr"] < 0.5))
thin_visible = (df["word_count"] > 0) & (df["word_count"] < 1200) & (df["impressions_90d"] >= 250)

print("stale_visible pages:  ", stale_visible.sum())
print("low_ctr_visible pages:", low_ctr_visible.sum())
print("thin_visible pages:   ", thin_visible.sum())


stale_visible pages:   17
low_ctr_visible pages: 9759
thin_visible pages:    82


## 2. Build the ranked queue (writes the CSV)

Score = how many of the three reasons match (0, 1, or 2 in this data -- no page matches all
three), plus a visibility tiebreaker so that within the same reason count, the page with more
search demand surfaces first. No fitted weights, no label inside the formula -- an addable,
readable rule anyone can check by hand.

In [2]:
reason_count = stale_visible.astype(int) + low_ctr_visible.astype(int) + thin_visible.astype(int)
visibility_tiebreak = df["impressions_90d"].rank(pct=True)
df["rule_score"] = reason_count + 0.99 * visibility_tiebreak

def make_reason_codes(i):
    reasons = []
    if stale_visible[i]:
        reasons.append("stale_visible")
    if low_ctr_visible[i]:
        reasons.append("low_ctr_visible")
    if thin_visible[i]:
        reasons.append("thin_visible")
    return "|".join(reasons) if reasons else "general_refresh_review"

df["reason_codes"] = [make_reason_codes(i) for i in df.index]

def suggested_action(reasons):
    r = set(reasons.split("|"))
    if "thin_visible" in r:
        return "expand_and_refresh"
    if "low_ctr_visible" in r:
        return "refresh_and_review_ctr"
    if "stale_visible" in r:
        return "refresh"
    return "monitor"

df["suggested_action"] = df["reason_codes"].apply(suggested_action)
df["rank"] = df["rule_score"].rank(method="first", ascending=False).astype(int)

out_dir = Path("../outputs")
out_dir.mkdir(parents=True, exist_ok=True)
out_cols = ["content_id", "client_id", "rank", "rule_score", "reason_codes", "suggested_action",
            "impressions_90d", "avg_position", "ctr", "word_count", "days_since_last_update",
            "is_down"]
df[out_cols].sort_values("rank").to_csv(out_dir / "baseline_action_score.csv", index=False)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

print("base rate:", f"{df['is_down'].mean():.1%}")
print("P@20:", f"{precision_at_k(df['rule_score'], df['is_down'], 20):.3f}")
print("P@50:", f"{precision_at_k(df['rule_score'], df['is_down'], 50):.3f}")
print("wrote", out_dir / "baseline_action_score.csv")


base rate: 54.2%
P@20: 0.850
P@50: 0.720
wrote ..\outputs\baseline_action_score.csv


## 3. Top-20 review

Built programmatically below (action, reason code, a rough confidence note, and what would make
each wrong) so it matches the CSV exactly rather than being hand-typed. Confidence = number of
reasons matched (2 = high, 1 = medium). Across all 28 pages that match two reasons at once, 25
(89.3%) are genuinely declining -- the top of the queue is dominated by that group.

In [3]:
caveats = {
    "stale_visible": "wrong if the staleness is deliberate (an evergreen reference page), not neglect",
    "low_ctr_visible": "wrong if the CTR gap is seasonal noise, not a real title/snippet problem",
    "thin_visible": "wrong if the format does not need more words (e.g. a quick-answer page)",
}

def caveat_for(reasons):
    parts = [caveats[r] for r in reasons.split("|") if r in caveats]
    return "; ".join(parts) if parts else "no specific flag matched -- lowest-confidence pick"

top20 = df.sort_values("rank").head(20).copy()
top20["confidence"] = top20["reason_codes"].apply(
    lambda r: "high" if r.count("|") >= 1 else "medium")
top20["what_would_make_it_wrong"] = top20["reason_codes"].apply(caveat_for)

top20[["rank", "content_id", "suggested_action", "reason_codes", "confidence",
       "what_would_make_it_wrong", "is_down"]]


,rank,content_id,suggested_action,reason_codes,confidence,what_would_make_it_wrong,is_down
16751,1,content_cf56e2e2e282,refresh_and_review_ctr,stale_visible|low_ctr_visible,high,wrong if the staleness is deliberate (an everg...,1
21268,2,content_0a91db491d14,refresh_and_review_ctr,stale_visible|low_ctr_visible,high,wrong if the staleness is deliberate (an everg...,1
12045,3,content_c2d929d83eaa,refresh_and_review_ctr,stale_visible|low_ctr_visible,high,wrong if the staleness is deliberate (an everg...,1
22332,4,content_7c2869a87415,expand_and_refresh,low_ctr_visible|thin_visible,high,"wrong if the CTR gap is seasonal noise, not a ...",1
4071,5,content_18368918809f,expand_and_refresh,low_ctr_visible|thin_visible,high,"wrong if the CTR gap is seasonal noise, not a ...",1
5327,6,content_fe16a55cd13d,refresh_and_review_ctr,stale_visible|low_ctr_visible,high,wrong if the staleness is deliberate (an everg...,1
27119,7,content_f83ea742e459,expand_and_refresh,low_ctr_visible|thin_visible,high,"wrong if the CTR gap is seasonal noise, not a ...",0
6318,8,content_53da2fba59f6,expand_and_refresh,low_ctr_visible|thin_visible,high,"wrong if the CTR gap is seasonal noise, not a ...",1
29814,9,content_b4a3d5ad5ff4,expand_and_refresh,low_ctr_visible|thin_visible,high,"wrong if the CTR gap is seasonal noise, not a ...",1
20202,10,content_bdc45d0800d7,expand_and_refresh,low_ctr_visible|thin_visible,high,"wrong if the CTR gap is seasonal noise, not a ...",1


## 4. Weak picks + leakage check

Three of the top 20 are not actually declining (one `up`, two `stable`) -- all three matched
BOTH `low_ctr_visible` and `thin_visible`, with no `stale_visible`. The rule's blind spot: a page
can look exactly like a decliner (thin, weak CTR, high demand) while its trend is actually flat
or improving. That is the honest cost of a 2-reason threshold here -- 89.3% right on that group
is good, not perfect. Also confirming below that the score itself never touches the label or a
future-window column.

In [4]:
weak = top20[top20["is_down"] == 0]
print(f"{len(weak)} of the top 20 are NOT actually declining:")
print(weak[["content_id", "reason_codes", "impressions_90d", "ctr", "word_count"]].to_string(index=False))

banned = {"trend_direction", "trend_pct", "impressions_last_30d", "clicks_last_30d",
          "sessions_last_30d", "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
          "provider_used", "model_used"}
score_inputs = {"days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count"}

print("\nScore built only from:", sorted(score_inputs))
print("Any banned column among the score inputs?", banned & score_inputs or "NONE (good)")


3 of the top 20 are NOT actually declining:
          content_id                 reason_codes  impressions_90d  ctr  word_count
content_f83ea742e459 low_ctr_visible|thin_visible             4116 0.19      1194.0
content_75568063406d low_ctr_visible|thin_visible             2150 0.28      1159.0
content_47b34667627e low_ctr_visible|thin_visible             1065 0.38      1162.0

Score built only from: ['avg_position', 'ctr', 'days_since_last_update', 'impressions_90d', 'word_count']
Any banned column among the score inputs? NONE (good)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.